# This notebook demonstrates a series of experiments conducted with ADAg on the CASPER dataset

## Step 0. Setup the experiment

In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import os
import pandas as pd
import numpy as np
from adag.models import load_anomaly_detector
from support.evaluate import evaluate
from spag.models import DeepForecaster

## Step 1. Load data 

### Test 

In [29]:
test = {}
data_folder = os.path.join('data', 'spag', 'input')
for state in os.listdir(data_folder):
    test[state] = {}
    state_path = os.path.join(data_folder, state, 'test')
    test[state]['data'] = np.load(os.path.join(state_path, 'data.npy'))
    test[state]['targets'] = np.load(os.path.join(state_path, 'targets.npy'))
    print(state, test[state]['data'].shape, test[state]['targets'].shape)

IMU_10Hz (700, 60, 9) (700, 15, 9)
IMU_2Hz (700, 60, 9) (700, 15, 9)
IMU_Acc_Gyro_20Hz (700, 60, 9) (700, 15, 9)
IMU_earthquake (700, 60, 9) (700, 15, 9)
IMU_hitting (700, 60, 9) (700, 15, 9)
IMU_magnet (700, 60, 9) (700, 15, 9)


### Predictions

In [30]:
predictions = {}
pred_dir = os.path.join('data', 'spag', 'output')
for state in os.listdir(pred_dir):
    predictions[state] = {}
    state_path = os.path.join(pred_dir, state)
    for model_name in os.listdir(state_path):
        predictions[state][model_name] = {}
        file_path = os.path.join(state_path, model_name, 'targets.npy')
        predictions[state][model_name]['targets'] = np.load(file_path)
        print(state, model_name, predictions[state][model_name]['targets'].shape)

IMU_10Hz gru_32_16_d005 (700, 15, 9)
IMU_10Hz gru_32_d005 (700, 15, 9)
IMU_10Hz lstm_32_16_d01 (700, 15, 9)
IMU_2Hz gru_32_16_d005 (700, 15, 9)
IMU_2Hz gru_32_d005 (700, 15, 9)
IMU_2Hz lstm_32_16_d01 (700, 15, 9)
IMU_Acc_Gyro_20Hz gru_32_16_d005 (700, 15, 9)
IMU_Acc_Gyro_20Hz gru_32_d005 (700, 15, 9)
IMU_Acc_Gyro_20Hz lstm_32_16_d01 (700, 15, 9)
IMU_earthquake gru_32_16_d005 (700, 15, 9)
IMU_earthquake gru_32_d005 (700, 15, 9)
IMU_earthquake lstm_32_16_d01 (700, 15, 9)
IMU_hitting gru_32_16_d005 (700, 15, 9)
IMU_hitting gru_32_d005 (700, 15, 9)
IMU_hitting lstm_32_16_d01 (700, 15, 9)
IMU_magnet gru_32_16_d005 (700, 15, 9)
IMU_magnet gru_32_d005 (700, 15, 9)
IMU_magnet lstm_32_16_d01 (700, 15, 9)


In [31]:
model_names = ['GRU-1', 'GRU-2', 'LSTM-2']
pred_mapping = {
    'GRU-1': 'gru_32_d005',
    'GRU-2': 'gru_32_16_d005',
    'LSTM-2': 'lstm_32_16_d01',
}

In [32]:
trues = {}
preds = {}
labels = {}
for normal_state in ['IMU_2Hz', 'IMU_10Hz', 'IMU_Acc_Gyro_20Hz']:
    trues[normal_state] = {}
    preds[normal_state] = {}
    labels[normal_state] = {}
    for model_name in model_names:
        print(normal_state, model_name)
        pred_values = []
        true_values = []
        true_labels = []
        model_path = os.path.join('models', 'spag', normal_state, f'{model_name}.keras')
        model = DeepForecaster(from_file=model_path)
        for state in [normal_state, 'IMU_earthquake', 'IMU_hitting', 'IMU_magnet']:
            if state in ['IMU_2Hz', 'IMU_10Hz', 'IMU_Acc_Gyro_20Hz']:
                y = 0
                pred_values.extend(predictions[state][pred_mapping[model_name]]['targets'])
            else:
                y = 1
                pred_values.extend(model.forecasting(data=test[state]['data'], verbose=0))
            true_labels.extend([y for _ in range(len(test[state]['targets']))])
            true_values.extend(test[state]['targets'])
        trues[normal_state][model_name] = np.array(true_values)
        preds[normal_state][model_name] = np.array(pred_values)
        labels[normal_state][model_name] = true_labels

IMU_2Hz GRU-1
IMU_2Hz GRU-2
IMU_2Hz LSTM-2
IMU_10Hz GRU-1
IMU_10Hz GRU-2
IMU_10Hz LSTM-2
IMU_Acc_Gyro_20Hz GRU-1
IMU_Acc_Gyro_20Hz GRU-2
IMU_Acc_Gyro_20Hz LSTM-2


## Step 2. Anomaly detection

In [16]:
detectors = {}
detectors_path = os.path.join('models', 'adag')
for state in os.listdir(detectors_path):
    detectors[state] = {}
    state_path = os.path.join(detectors_path, state)
    for model_name in os.listdir(state_path):
        print(state, model_name)
        model_path = os.path.join(state_path, model_name)
        detectors[state][model_name.replace('.json', '')] = load_anomaly_detector(model_path)  
        print('Detector parameters:', detectors[state][model_name.replace('.json', '')].__dict__)

IMU_10Hz GRU-1.json
Detector parameters: {'percentile': 0.8, 'classes': {0: [0, 1, 2], 1: [3, 4, 5]}, 'alpha': 0.4, 'beta': 0.3, 'gamma': 0.2, 'shape': 0.8617363077575857, 'loc': 0, 'scale': 0.0005278758151813346}
IMU_10Hz GRU-2.json
Detector parameters: {'percentile': 0.8, 'classes': {0: [0, 1, 2], 1: [3, 4, 5]}, 'alpha': 0.4, 'beta': 0.3, 'gamma': 0.2, 'shape': 0.845128589018625, 'loc': 0, 'scale': 0.0005343971945116601}
IMU_10Hz LSTM-2.json
Detector parameters: {'percentile': 0.8, 'classes': {0: [0, 1, 2], 1: [3, 4, 5]}, 'alpha': 0.4, 'beta': 0.3, 'gamma': 0.2, 'shape': 0.809519147237181, 'loc': 0, 'scale': 0.0006873870227967079}
IMU_2Hz GRU-1.json
Detector parameters: {'percentile': 0.99, 'classes': {0: [0, 1, 2], 1: [3, 4, 5]}, 'alpha': 0.4, 'beta': 0.3, 'gamma': 0.2, 'shape': 4.300503714288001, 'loc': 0, 'scale': 0.00038963380053421684}
IMU_2Hz GRU-2.json
Detector parameters: {'percentile': 0.95, 'classes': {0: [0, 1, 2], 1: [3, 4, 5]}, 'alpha': 0.4, 'beta': 0.3, 'gamma': 0.2, 's

### Combination of labels

In [33]:
classes = {'IMU_2Hz': [0, 3, 4, 5], 'IMU_10Hz': [1, 3, 4, 5], 'IMU_Acc_Gyro_20Hz': [2, 3, 4, 5]}

### Get data from SCAg agenets

In [34]:
probs_path = os.path.join('data', 'spag', 'results', 'probs.csv')
scag_results = pd.read_csv(probs_path)

In [35]:
anomaly_detection_results = {}
for normal_state, model_detectors in detectors.items():
    anomaly_detection_results[normal_state] = {}
    for model_name, detector in model_detectors.items():
        anomaly_detection_results[normal_state][model_name] = {}
        
        for clf_model in ['ParametrizedMLP', 'ParametrizedCNN', 'ParametrizedTransformer']:
            print(normal_state, model_name, clf_model)
            anomaly_detection_results[normal_state][model_name][clf_model] = {}
            
            df = pd.concat([scag_results[(scag_results['pred_model'] == model_name) 
                     & (scag_results['clf_model'] == clf_model) 
                     & (scag_results['real_class'] == st)][:2000] for st in classes[normal_state]],
                           ignore_index=True)
 
            anomaly_detection_results[normal_state][model_name][clf_model]['simple'] = (
                detector.simply_run(true_data=trues[normal_state][model_name], 
                                    pred_data=preds[normal_state][model_name]))
            
            anomaly_detection_results[normal_state][model_name][clf_model]['corrected'] = (
                detector.run(true_data=trues[normal_state][model_name], pred_data=preds[normal_state][model_name],
                             current_labels=df['test_class'].values, pred_labels=df['pred_class'].values,
                             current_proba=df[[c for c in df.columns if ('t_class' in c) and ('prob' in c)]].values,
                             pred_proba=df[[c for c in df.columns if ('p_class' in c) and ('prob' in c)]].values))

IMU_10Hz GRU-1 ParametrizedMLP
IMU_10Hz GRU-1 ParametrizedCNN
IMU_10Hz GRU-1 ParametrizedTransformer
IMU_10Hz GRU-2 ParametrizedMLP
IMU_10Hz GRU-2 ParametrizedCNN
IMU_10Hz GRU-2 ParametrizedTransformer
IMU_10Hz LSTM-2 ParametrizedMLP
IMU_10Hz LSTM-2 ParametrizedCNN
IMU_10Hz LSTM-2 ParametrizedTransformer
IMU_2Hz GRU-1 ParametrizedMLP
IMU_2Hz GRU-1 ParametrizedCNN
IMU_2Hz GRU-1 ParametrizedTransformer
IMU_2Hz GRU-2 ParametrizedMLP
IMU_2Hz GRU-2 ParametrizedCNN
IMU_2Hz GRU-2 ParametrizedTransformer
IMU_2Hz LSTM-2 ParametrizedMLP
IMU_2Hz LSTM-2 ParametrizedCNN
IMU_2Hz LSTM-2 ParametrizedTransformer
IMU_Acc_Gyro_20Hz GRU-1 ParametrizedMLP
IMU_Acc_Gyro_20Hz GRU-1 ParametrizedCNN
IMU_Acc_Gyro_20Hz GRU-1 ParametrizedTransformer
IMU_Acc_Gyro_20Hz GRU-2 ParametrizedMLP
IMU_Acc_Gyro_20Hz GRU-2 ParametrizedCNN
IMU_Acc_Gyro_20Hz GRU-2 ParametrizedTransformer
IMU_Acc_Gyro_20Hz LSTM-2 ParametrizedMLP
IMU_Acc_Gyro_20Hz LSTM-2 ParametrizedCNN
IMU_Acc_Gyro_20Hz LSTM-2 ParametrizedTransformer


### Connected with Table 9

In [36]:
results = pd.DataFrame(columns=['normal_state', 'pred_model', 'clf_model', 'ACC', 'ACC*', 'P', 'P*', 'R', 'R*', 'F1', 'F1*', 'FNR', 'FPR', 'FNR*', 'FPR*'])
i = 0
for normal_state, state_results in anomaly_detection_results.items():
    for reg_model, reg_val in state_results.items():
        for clf_model, val in reg_val.items():
            for mode in ['simple', 'corrected']:
                results.loc[i, 'normal_state'] = normal_state
                results.loc[i, 'pred_model'] = reg_model
                results.loc[i, 'clf_model'] = clf_model
                evals = evaluate(labels[normal_state][reg_model], val[mode]['labels'], balanced=True)
                if mode == 'simple':
                    results.loc[i, ['ACC', 'P', 'R', 'F1', 'FNR', 'FPR']] = list(evals.values())
                else:
                    results.loc[i, ['ACC*', 'P*', 'R*', 'F1*', 'FNR*', 'FPR*']] = list(evals.values())
            i = i + 1
results

,normal_state,pred_model,clf_model,ACC,ACC*,P,P*,R,R*,F1,F1*,FNR,FPR,FNR*,FPR*
0,IMU_10Hz,GRU-1,ParametrizedMLP,0.831,0.5,0.783,1.0,0.914,0.0,0.844,0.0,0.253,0.086,0.0,1.0
1,IMU_10Hz,GRU-1,ParametrizedCNN,0.834,0.5,0.784,1.0,0.92,0.0,0.847,0.0,0.253,0.08,0.0,1.0
2,IMU_10Hz,GRU-1,ParametrizedTransformer,0.833,0.5,0.784,1.0,0.919,0.0,0.846,0.0,0.253,0.081,0.0,1.0
3,IMU_10Hz,GRU-2,ParametrizedMLP,0.811,0.5,0.769,1.0,0.889,0.0,0.824,0.0,0.267,0.111,0.0,1.0
4,IMU_10Hz,GRU-2,ParametrizedCNN,0.816,0.5,0.771,1.0,0.9,0.0,0.831,0.0,0.267,0.1,0.0,1.0
5,IMU_10Hz,GRU-2,ParametrizedTransformer,0.812,0.5,0.769,1.0,0.891,0.0,0.826,0.0,0.267,0.109,0.0,1.0
6,IMU_10Hz,LSTM-2,ParametrizedMLP,0.791,0.5,0.763,1.0,0.846,0.0,0.802,0.0,0.263,0.154,0.0,1.0
7,IMU_10Hz,LSTM-2,ParametrizedCNN,0.78,0.5,0.758,1.0,0.823,0.0,0.789,0.0,0.263,0.177,0.0,1.0
8,IMU_10Hz,LSTM-2,ParametrizedTransformer,0.799,0.5,0.766,1.0,0.86,0.0,0.81,0.0,0.263,0.14,0.0,1.0
9,IMU_2Hz,GRU-1,ParametrizedMLP,1.0,0.5,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


## This concludes the demonstration for the CASPER dataset.